In [28]:
import os
import json
import shutil
import numpy as np
from copy import deepcopy

In [29]:
# Original train folder with images and COCO JSON
original_train_dir = "../data/synthetic_mnist_split/train"
original_coco_json = os.path.join(original_train_dir, "_annotations.coco.json")

# Output split folders
split_root = "../data/synthetic_mnist_split_split"
train_dir = os.path.join(split_root, "train")
val_dir   = os.path.join(split_root, "val")
test_dir  = os.path.join(split_root, "test")

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Split ratios
train_ratio = 0.7
val_ratio   = 0.15
test_ratio  = 0.15

In [30]:
with open(original_coco_json) as f:
    coco_data = json.load(f)

images = coco_data["images"]
annotations = coco_data["annotations"]

print(f"Total images: {len(images)}")
print(f"Total annotations: {len(annotations)}")

Total images: 112
Total annotations: 1860


In [31]:
np.random.seed(42)
shuffled_images = deepcopy(images)
np.random.shuffle(shuffled_images)

num_total = len(shuffled_images)
num_train = int(train_ratio * num_total)
num_val   = int(val_ratio * num_total)

train_images = shuffled_images[:num_train]
val_images   = shuffled_images[num_train:num_train+num_val]
test_images  = shuffled_images[num_train+num_val:]

In [32]:
def filter_annotations(img_list, all_annotations):
    img_ids = set([img["id"] for img in img_list])
    filtered = [ann for ann in all_annotations if ann["image_id"] in img_ids]
    return filtered

In [33]:
def copy_images(img_list, src_folder, dst_folder):
    os.makedirs(dst_folder, exist_ok=True)
    for img in img_list:
        src = os.path.join(src_folder, img["file_name"])
        dst = os.path.join(dst_folder, img["file_name"])
        shutil.copy(src, dst)
copy_images(train_images, original_train_dir, train_dir)
copy_images(val_images, original_train_dir, val_dir)
copy_images(test_images, original_train_dir, test_dir)
        

In [34]:
def write_coco_json(images_list, annotations_list, categories, out_path):
    coco_dict = {
        "info": coco_data.get("info", {}),
        "licenses": coco_data.get("licenses", []),
        "images": images_list,
        "annotations": annotations_list,
        "categories": categories
    }
    with open(out_path, "w") as f:
        json.dump(coco_dict, f, indent=4)

In [35]:
categories = coco_data["categories"]

# Train JSON
train_anns = filter_annotations(train_images, annotations)
write_coco_json(train_images, train_anns, categories, os.path.join(train_dir, "_annotations.coco.json"))

# Val JSON
val_anns = filter_annotations(val_images, annotations)
write_coco_json(val_images, val_anns, categories, os.path.join(val_dir, "_annotations.coco.json"))

# Test JSON
test_anns = filter_annotations(test_images, annotations)
write_coco_json(test_images, test_anns, categories, os.path.join(test_dir, "_annotations.coco.json"))

In [36]:
print(f"Train images: {len(train_images)}, annotations: {len(train_anns)}")
print(f"Validation images: {len(val_images)}, annotations: {len(val_anns)}")
print(f"Test images: {len(test_images)}, annotations: {len(test_anns)}")
print("Dataset split and COCO JSON files created successfully!")

Train images: 78, annotations: 1307
Validation images: 16, annotations: 265
Test images: 18, annotations: 288
Dataset split and COCO JSON files created successfully!


In [37]:
import os
import re
import json

# Base folder containing train, val, test
base_dir = "../data/synthetic_mnist_split_split"
splits = ["train", "val", "test"]

for split in splits:
    split_dir = os.path.join(base_dir, split)
    json_path = os.path.join(split_dir, "_annotations.coco.json")
    
    # Load COCO JSON if it exists
    if os.path.exists(json_path):
        with open(json_path) as f:
            coco = json.load(f)
    else:
        coco = None
    
    # Process all images in folder
    for f in os.listdir(split_dir):
        if not f.endswith(".png"):
            continue
        # Create new name by removing _png.rf.<hash>
        new_name = re.sub(r'_png\.rf\..*$', '.png', f)
        old_path = os.path.join(split_dir, f)
        new_path = os.path.join(split_dir, new_name)
        os.rename(old_path, new_path)
        
        # Update JSON if available
        if coco:
            for img in coco["images"]:
                if img["file_name"] == f:
                    img["file_name"] = new_name
    
    # Save updated JSON
    if coco:
        with open(json_path, "w") as f:
            json.dump(coco, f, indent=4)
        print(f"Updated JSON for {split} split.")

print("All images renamed and JSON updated for train, val, and test splits.")

Updated JSON for train split.
Updated JSON for val split.
Updated JSON for test split.
All images renamed and JSON updated for train, val, and test splits.


In [48]:
import os
import json
import random
import shutil

# Original dataset
dataset_dir = "../data/synthetic_mnist_split"
train_dir = os.path.join(dataset_dir, "train")
json_file = os.path.join(train_dir, "_annotations.coco.json")
images_dir = train_dir  # all images are here

# YOLO output folder
yolo_dir = "../data/mnist_yolo_training"

# Load COCO JSON
with open(json_file, "r") as f:
    coco = json.load(f)

# Split ratios
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

# Shuffle images
images = coco["images"]
random.seed(42)
random.shuffle(images)

n_total = len(images)
n_train = int(train_ratio * n_total)
n_val = int(val_ratio * n_total)

train_imgs = images[:n_train]
val_imgs = images[n_train:n_train + n_val]
test_imgs = images[n_train + n_val:]

splits = {"train": train_imgs, "val": val_imgs, "test": test_imgs}

# Create directories for images and labels
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(yolo_dir, split), exist_ok=True)
    os.makedirs(os.path.join(yolo_dir, "labels", split), exist_ok=True)

# Track all zero-based class IDs
all_class_ids = set()

# Process each split
for split_name, split_images in splits.items():
    for img_info in split_images:
        img_id = img_info["id"]
        file_name = img_info["file_name"]

        # Copy image to YOLO folder
        src_img_path = os.path.join(images_dir, file_name)
        dst_img_path = os.path.join(yolo_dir, split_name, file_name)
        if os.path.exists(src_img_path):
            shutil.copy2(src_img_path, dst_img_path)
        else:
            print(f"Image not found, skipping: {file_name}")
            continue

        # Create YOLO label file
        label_file = os.path.join(
            yolo_dir, "labels", split_name, os.path.splitext(file_name)[0] + ".txt"
        )
        anns = [ann for ann in coco["annotations"] if ann["image_id"] == img_id]

        with open(label_file, "w") as f:
            for ann in anns:
                x_min, y_min, w, h = ann["bbox"]
                img_w, img_h = img_info["width"], img_info["height"]
                x_center = (x_min + w / 2) / img_w
                y_center = (y_min + h / 2) / img_h
                w_norm = w / img_w
                h_norm = h / img_h

                # Convert to zero-based class ID
                class_id = ann["category_id"] - 1
                if 0 <= class_id <= 10:  # valid classes
                    all_class_ids.add(class_id)
                    f.write(f"{class_id} {x_center} {y_center} {w_norm} {h_norm}\n")
                else:
                    print(f"Skipping invalid class_id {ann['category_id']} in {file_name}")

# Show all zero-based class IDs
print("All class IDs in dataset (zero-based):", sorted(all_class_ids))

print("YOLO training folder created successfully!")
print(f"Images: {os.path.join(yolo_dir, 'train')}, {os.path.join(yolo_dir, 'val')}, {os.path.join(yolo_dir, 'test')}")
print(f"Labels: {os.path.join(yolo_dir, 'labels', 'train')}, {os.path.join(yolo_dir, 'labels', 'val')}, {os.path.join(yolo_dir, 'labels', 'test')}")

All class IDs in dataset (zero-based): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
YOLO training folder created successfully!
Images: ../data/mnist_yolo_training\train, ../data/mnist_yolo_training\val, ../data/mnist_yolo_training\test
Labels: ../data/mnist_yolo_training\labels\train, ../data/mnist_yolo_training\labels\val, ../data/mnist_yolo_training\labels\test


In [49]:
labels_dir = os.path.join(yolo_dir, "labels")

for split in ["train", "val", "test"]:
    img_folder = os.path.join(yolo_dir, split)
    label_folder = os.path.join(labels_dir, split)
    
    img_files = [f for f in os.listdir(img_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    label_files = [f for f in os.listdir(label_folder) if f.lower().endswith('.txt')]
    
    # Check counts
    print(f"{split} images: {len(img_files)}, labels: {len(label_files)}")
    
    # Check for mismatch
    missing_labels = [f for f in img_files if os.path.splitext(f)[0] + ".txt" not in label_files]
    print(f"{split} images without labels:", missing_labels[:5])  # show first 5

train images: 89, labels: 89
train images without labels: []
val images: 11, labels: 11
val images without labels: []
test images: 12, labels: 12
test images without labels: []


In [46]:
# Write corresponding YOLO labels
label_file = os.path.join(yolo_dir, "labels", split_name, os.path.splitext(file_name)[0] + ".txt")
anns = [ann for ann in coco["annotations"] if ann["image_id"] == img_id]

with open(label_file, "w") as f:   # <-- make sure this 'with' wraps all writes
    for ann in anns:
        x_min, y_min, w, h = ann["bbox"]
        img_w, img_h = img_info["width"], img_info["height"]
        x_center = (x_min + w / 2) / img_w
        y_center = (y_min + h / 2) / img_h
        w_norm = w / img_w
        h_norm = h / img_h

        # Subtract 1 from category_id if your class IDs start at 1
        class_id = ann["category_id"] - 1

        f.write(f"{class_id} {x_center} {y_center} {w_norm} {h_norm}\n")

In [50]:
data_path = "../data/mnist_yolo_training"

for split in ["train", "val", "test"]:
    img_folder = os.path.join(data_path, split)
    label_folder = os.path.join(data_path, "labels", split)

    # Create the label folder if it doesn't exist
    os.makedirs(label_folder, exist_ok=True)

    # Optional: check if img_folder exists
    if not os.path.exists(img_folder):
        print(f"Image folder does not exist, creating: {img_folder}")
        os.makedirs(img_folder, exist_ok=True)

    # Copy all label files next to images
    for txt_file in os.listdir(label_folder):
        src = os.path.join(label_folder, txt_file)
        dst = os.path.join(img_folder, txt_file)
        shutil.copy(src, dst)

    print(f"Copied labels for {split} split.")

Copied labels for train split.
Copied labels for val split.
Copied labels for test split.
